# Special SQL in `duckdb`

This is again just the prologue to allow the notebooks running standalone in a local Jupyter installation and in Google Colab.

In [ ]:
import os
try:
    import google.colab
    IN_COLAB = True
    if not os.path.isdir("/content/oreilly-duckdb"):
        os.system("git clone https://github.com/datanizing/oreilly-duckdb")
    os.system("pip install -U duckdb")
    data = "/content/oreilly-duckdb/data"
except:
    IN_COLAB = False
    data = "data"

In [ ]:
import duckdb

## Create a temporary table

When you want to analyze data, the data if often not in the
correct format. Take the Australian weather data as an example.
A lot of measurements are available for each date. But the
granularity of the `Date` field is *per day*.

If you want to calculate some statistics on a *per month* or
even *per year* level, you can change that granularity. In
normal relational databases, you would work with a view. This
is also possible in `duckdb`, however you will more often choose
a *temporary table*. A table is always temporary if you are
working in memory mode, otherwise you have to use the attribute
`TEMPORARY` (which does not hurt in memory mode anyway).

`duckdb` makes it even more convenient and support the
`CREATE OR REPLACE` functionality. This way, you don't have to
`DROP` the table first. Try it, you can run the statement
over and over again:

In [ ]:
duckdb.sql(f"CREATE OR REPLACE TEMPORARY TABLE weather AS \
               SELECT * FROM '{data}/weatherAUS.csv.zst'")

In [ ]:
duckdb.sql(f"CREATE OR REPLACE TEMPORARY TABLE rainfall AS \
               SELECT Location, Date, date_trunc('year', Date) AS Year, \
                      date_trunc('month', Date) as Month, Rainfall \
               FROM weather")

For selecting, temporary tables work exactly in the same way
as "normal" tables:

In [ ]:
duckdb.sql("SELECT * FROM weather").pl()

In [ ]:
duckdb.sql("SELECT * FROM rainfall").pl()

Note that `Date` and `Year` still have the correct data type but
a different granularity. If you want to have a shorter version,
you could use `strftime` for this:

In [ ]:
duckdb.sql(f"SELECT Location, Date, strftime('%Y', Date) AS Year, \
                    strftime('%Y-%m', Date) as Month, Rainfall \
             FROM '{data}/weatherAUS.csv.zst'").pl()

## `PIVOT`

It is not too difficult to get the sum of the rainfall per `Location` and `Year`
or `Month` now, this can be achieved very easily with `GROUP BY`:

In [ ]:
duckdb.sql("SELECT Location, Year, sum(Rainfall) FROM Rainfall GROUP BY ALL").pl()

Often, you might want to have this data in a *different shape*.
Currently, you see the so-called *long* format, but a *broad* format
might be what you want for plotting the data of the individual cities.

`pandas` and `polars` have functions for this called `pivot`:

In [ ]:
duckdb.sql("SELECT Location, Year, sum(Rainfall) FROM Rainfall GROUP BY ALL").pl().pivot("Location", index="Year")

`duckdb` can do this directly. Why would you want to do that if you
have the whole Python ecosystem behind you? There are several
scenarios where this is helpful, especially if you want to save
intermediate results in a (persisted) table or in a `parquet`file.

This way, the (possibly) large data does not need to be transfered
to Python. In the example above, individual columns can be selected
if you are only interested in these cities. Let's try:

In [ ]:
duckdb.sql("PIVOT rainfall ON Location USING sum(Rainfall) GROUP BY YEAR ORDER BY YEAR").pl()

Great, looks exactly identical. Now let's save this in a table. Remember,
we are in memory mode therefore it does not get persisted:

In [ ]:
duckdb.sql("CREATE OR REPLACE TABLE rainfall_year AS \
                 PIVOT rainfall ON Location USING sum(Rainfall) GROUP BY year ORDER BY year")

Now it is easy to just select a few columns and plot them. Plotting multiple 
columns is much easier in `pandas`. `duckdb` allows you to change between
the frameworks with very little effort:

In [ ]:
duckdb.sql("SELECT year, Canberra, Perth, Sydney FROM rainfall_year").df().set_index("year").plot()

If you like to see seasonal changes, you have to go to month granularity.
It is not difficult at all! Create the pivot table first:

In [ ]:
duckdb.sql("CREATE OR REPLACE TABLE rainfall_month AS \
                 PIVOT rainfall ON Location USING sum(Rainfall) GROUP BY month ORDER BY month")

Now adjust the `SELECT` statement and the index:

In [ ]:
duckdb.sql("SELECT month, Canberra, Perth, Sydney FROM rainfall_month").df().set_index("month").plot()

## Window Functions

Window functions are a very nice addition to SQL. They allow you to
"peek" inside a window of the current row. This is very useful to
calculate sums while still seeing the line items. 

It's a bit easier to see what's happening if we reduce the time interval:

In [ ]:
duckdb.sql("SELECT Location, Date, Rainfall, \
                   sum(Rainfall) OVER (PARTITION BY Location) AS TotalRainfall, \
                   row_number() OVER (PARTITION BY Location ORDER BY Date) AS row\
            FROM rainfall WHERE Date>='2015-09-01' AND date<'2015-10-01'").pl().head(34)

As you can see, we stil have the detail level in this table,
but also the sum. It would be easy to calculate the fraction
of rain on each day etc.

Window functions can also be used to calculate *running averages*
which is super useful with financial data, but also with our
example weather data:

In [ ]:
duckdb.sql("SELECT Location, Date, Rainfall, \
                   sum(Rainfall) OVER (PARTITION BY Location) AS TotalRainfall, \
                   sum(Rainfall) OVER (PARTITION BY Location ORDER BY Date) AS TotalRainfallUntilToday, \
                   row_number() OVER (PARTITION BY Location ORDER BY Date) AS row\
            FROM rainfall WHERE Date>='2015-09-01' AND date<'2015-10-01'").pl().head(34)

When using window functions, remember that the number of results
does not change. This is in sharp contrast to aggregations which
(almost) always reduces the number of results.

## Sophisticated aggregations

Often, `GROUP`ing is very useful, but you have to do it over and
over again to see the different aggregations. `duckdb` and other
modern SQL dialects solve this with `GROUPING SETS`. Grouping is
then performed not just with respect to a single column, using
many instead. An example with a shorter time interval for
better illustraction shows how this works:

In [ ]:
duckdb.sql("SELECT Location, date_trunc('month', Date) AS month, \
                   date_trunc('year', Date) AS year, avg(MinTemp) \
            FROM weather WHERE year<'2010-01-01' \
            GROUP BY GROUPING SETS ((Location, year, month), (Location, year), Location, ()) \
            ORDER BY Location NULLS FIRST, month NULLS FIRST, year NULLS FIRST").pl()

As you can see, the data has been grouped with respect to the totals first,
the `Location` without `year` and `month` first, then `Location` and `year`
and finally with respect to all fields.

Very frequently, you need this kind of aggregation sets, first grouped
by all fields, then always taking away the last field. For this scenario,
`duckdb` has a function which accomplishes exactly that called `ROLLUP`:

In [ ]:
duckdb.sql("SELECT Location, date_trunc('month', Date) AS month, \
                   date_trunc('year', Date) AS year, avg(MinTemp) \
            FROM weather WHERE year<'2010-01-01' \
            GROUP BY ROLLUP(Location, year, month) \
            ORDER BY Location NULLS FIRST, year NULLS FIRST, month NULLS FIRST").pl()

The result is exactly identical to the one before. 

If you have worked with datawarehouses before, you have probably
encountered a `CUBE`. This phrase builds all possible combinations
resulting in a much longer result set.

`duckdb`support also this kind of multi-aggregation
in its `SQL` dialect:

In [ ]:
duckdb.sql("SELECT Location, date_trunc('month', Date) AS month, \
                   date_trunc('year', Date) AS year, avg(MinTemp) \
            FROM weather WHERE year<'2010-01-01' \
            GROUP BY CUBE(Location, year, month) \
            ORDER BY Location NULLS FIRST, month NULLS FIRST, year NULLS FIRST").pl()